In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as colors

import numpy as np

import cartopy.crs as ccrs

import torch
import torch.nn as nn
import torch.utils.data as data
import torch_geometric
from torch.nn import Sequential as Seq, Linear, ReLU

import sys

sys.path.append("../src/")
from utils.data_utils import *
from utils.climate_utils import *
from utils.subgrid_utils import *
from utils.train_utils import *

from matplotlib.animation import FuncAnimation
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader
import os
import random

/ext3/miniconda3/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Init
args = {}
exp_num_in = "3"
exp_num_extra = "12"
exp_num_out = "2"
region = "Gulf_Stream_Ext"
network = "Transformer"
N_samples = 40  # 4000
N_val = 3  # 300
N_test = 5  # 500
factor = 10
interval = 1  # 2
hist = 0
lag = 1
lam = 50
steps = 2
Nb = 4
lateral = True
load = False
save_model = True
epochs = 8
batch_size = 6
rand_seed = 1
device = "cuda"
step_weights = [
    [1],
    [0.2, 0.8],
    [0.2, 0.2, 0.6],
    [0, 0, 0.2, 0.8],
    [0, 0, 0.1, 0.2, 0.7],
    [0, 0, 0, 0.1, 0.2, 0.7],
    [0, 0, 0, 0, 0.1, 0.2, 0.7],
]
step_lrs = [1e-4, 5e-5, 1e-5, 1e-5, 5e-6, 5e-6, 5e-6]
inpt_dict = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "ur", "vr"],
    "3": ["um", "vm", "Tm"],
    "4": ["um", "vm", "ur", "vr", "Tm", "Tr"],
    "5": ["ur", "vr"],
    "6": ["ur", "vr", "Tr"],
    "7": ["Tm"],
    "8": ["Tm", "Tr"],
    "9": ["u", "v"],
    "10": ["u", "v", "T"],
    "11": ["tau_u", "tau_v"],
    "12": ["tau_u", "tau_v", "t_ref"],
}
extra_dict = {
    "1": ["ur", "vr"],
    "2": ["ur", "vr", "Tm"],
    "3": ["Tm"],
    "4": ["ur", "vr", "Tm", "Tr"],
    "5": [],
    "6": ["um", "vm"],
    "7": ["um", "vm", "Tm"],
    "8": ["um", "vm", "Tm", "Tr"],
    "9": ["ur", "vr", "tau_u", "tau_v"],
    "10": ["tau_u", "tau_v"],
    "11": ["t_ref"],
    "12": ["tau_u", "tau_v", "t_ref"],
    "13": ["ur", "vr", "Tr", "tau_u", "tau_v", "t_ref"],
}
out_dict = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "Tm"],
    "3": ["ur", "vr"],
    "4": ["ur", "vr", "Tr"],
    "5": ["u", "v"],
    "6": ["u", "v", "T"],
}

# Set seed
torch.manual_seed(rand_seed)
random.seed(rand_seed)
np.random.seed(rand_seed)

# Env variables
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "1324"

# if region == "Quiescent":
#     interval = 1
if N_samples > 2000:
    interval = 1

s_train = lag * hist  # 1*0=0
e_train = s_train + N_samples * interval  # 0 + 4000*1 = 4000
e_test = e_train + interval * N_val  # 4000 + 1*300 = 4300

In [3]:
# Str video init
if len(sys.argv) > 4:
    n_cond = int((len(sys.argv) - 4) / 2)

str_video = ""

try:
    for i in range(n_cond):
        if type(globals()[sys.argv[int(4 + i * 2)]]) == str:
            temp = str(sys.argv[int(5 + i * 2)])
            exec(sys.argv[int(4 + i * 2)] + "= temp")
            if sys.argv[int(4 + i * 2)] == "network":
                continue
            str_video += "_" + sys.argv[int(4 + i * 2)] + "_" + sys.argv[int(5 + i * 2)]
        elif type(globals()[sys.argv[int(4 + i * 2)]]) == int:
            exec(
                sys.argv[int(4 + i * 2)] + "=" + "int(" + sys.argv[int(5 + i * 2)] + ")"
            )
            str_video += "_" + sys.argv[int(4 + i * 2)] + "_" + sys.argv[int(5 + i * 2)]
    print(str_video)
except:
    print("no cond")

str_video += "_Lateral_Data_025_no_smooth"

no cond


In [4]:
# Region init
if region == "Kuroshio":
    lat = [15, 41]
    lon = [-215, -185]
elif region == "Kuroshio_Ext":
    lat = [5, 50]
    lon = [-250, -175]
elif region == "Gulf_Stream":
    lat = [25, 50]
    lon = [-70, -35]
elif region == "Gulf_Stream_Ext":
    lat = [27, 50]
    lon = [-82, -35]
elif region == "Tropics":
    lat = [-5, 25]
    lon = [-95, -65]
elif region == "Tropics_Ext":
    lat = [-5, 25]
    lon = [-115, -45]
elif region == "South_America":
    lat = [-60, -30]
    lon = [-70, -35]
elif region == "Africa":
    lat = [-50, -20]
    lon = [5, 45]
elif region == "Africa_Ext":
    lat = [-55, -15]
    lon = [-5, 55]
elif region == "Quiescent":
    lat = [-42.5, -17.5]
    lon = [-155, -120]
elif region == "Quiescent_Ext":
    lat = [-55, -10]
    lon = [-170, -110]
elif region == "Pacific":
    lat = [-35, 35]
    lon = [-230, -80]
elif region == "Indian":
    lat = [-30, 28]
    lon = [30, 79]

In [5]:
# Getting input, extra input and output
inputs = inpt_dict[exp_num_in]
extra_in = extra_dict[exp_num_extra]
outputs = out_dict[exp_num_out]

str_in = "".join([i + "_" for i in inputs])
str_ext = "".join([i + "_" for i in extra_in])
str_out = "".join([i + "_" for i in outputs])

print("inputs: " + str_in)
print("extra inputs: " + str_ext)
print("outputs: " + str_out)

N_atm = len(extra_in)
N_in = len(inputs)
N_extra = N_atm + N_in
N_out = len(outputs)

num_in = int((hist + 1) * N_in + N_extra)
print("Number of inputs: ", num_in)  # 3 (ocean speeds + ocean temp)(t) +
# 3 (atm wind stresses + atm temp)(t) +
# 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
print("Number of outputs: ", N_out)

inputs: um_vm_Tm_
extra inputs: tau_u_tau_v_t_ref_
outputs: um_vm_Tm_
Number of inputs:  9
Number of outputs:  3


In [6]:
# Area selection
grids = xr.open_dataset("/scratch/as15415/Data/CM2x_grids/Grid_cm25_Vertices.nc")
if region == "global_25":
    grids = xr.open_dataset("/scratch/as15415/Data/CM2x_grids/Grid_cm25_Vertices.nc")

elif "global" in region:
    grids = coarse_grid(grids, factor)

else:
    grids = grids.sel(
        {"yu_ocean": slice(lat[0], lat[1]), "xu_ocean": slice(lon[0], lon[1])}
    )


area = torch.from_numpy(grids["area_C"].to_numpy()).to(device=device)
print(area.shape)

torch.Size([119, 189])


In [7]:
# Defining args
args["region"] = region
args["network"] = network
args["interval"] = interval
args["N_samples"] = N_samples
args["N_val"] = N_val
args["N_test"] = N_test
args["factor"] = factor
args["hist"] = hist
args["lag"] = lag
args["steps"] = steps
args["str_video"] = str_video
args["s_train"] = s_train
args["e_train"] = e_train
args["e_test"] = e_test
args["area"] = area
args["N_extra"] = N_extra
args["N_in"] = N_in
args["N_out"] = N_out
args["N_atm"] = N_atm
args["num_in"] = num_in
args["str_in"] = str_in
args["str_ext"] = str_ext
args["str_out"] = str_out
args["Nb"] = Nb
args["lateral"] = lateral
args["load"] = load
args["save_model"] = save_model
args["World_Size"] = int(torch.cuda.device_count())
args["epochs"] = epochs
args["batch_size"] = batch_size
args["lam"] = lam / 1000
args["step_weights"] = step_weights
args["step_lrs"] = step_lrs

In [8]:
# train_data = torch.load('/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/data/train_data_{0}.pt'.format('steps_'+str(args["steps"])+'_'+args["region"]+'_Test_in_' + args["str_in"] + 'ext_' + args["str_ext"] +'_out'+args['str_out']+'N_train_' + str(args["N_samples"]) + args["str_video"]))
# val_data = torch.load('/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/data/val_data_{0}.pt'.format('steps_'+str(args["steps"])+'_'+args["region"]+'_Test_in_' + args["str_in"] + 'ext_' + args["str_ext"] +'_out'+args['str_out']+'N_train_' + str(args["N_samples"]) + args["str_video"]))

train_data = torch.load(
    "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/.LOCAL/save_data/2024-03-06-save_data_test/data/train_data_steps_2_Gulf_Stream_Ext_Test_in_um_vm_Tm_ext_tau_u_tau_v_t_ref__outum_vm_Tm_N_train_40_Lateral_Data_025_no_smooth.pt"
)
val_data = torch.load(
    "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/.LOCAL/save_data/2024-03-06-save_data_test/data/val_data_steps_2_Gulf_Stream_Ext_Test_in_um_vm_Tm_ext_tau_u_tau_v_t_ref__outum_vm_Tm_N_train_40_Lateral_Data_025_no_smooth.pt"
)

train_sampler = torch.utils.data.distributed.DistributedSampler(
    train_data, num_replicas=1, rank=0
)

test_sampler = torch.utils.data.distributed.DistributedSampler(
    val_data, num_replicas=args["World_Size"], rank=0
)

train_loader = torch_geometric.loader.DataLoader(
    train_data, batch_size=args["batch_size"], sampler=train_sampler
)
test_loader = torch_geometric.loader.DataLoader(
    val_data, batch_size=args["batch_size"], sampler=test_sampler
)

In [9]:
def train_parallel_Dynamic(
    model, train_loader, N_in, N_extra, hist, loss_fn, optimizer, steps, weight, device
):
    mse = torch.nn.MSELoss()
    for data in train_loader:
        optimizer.zero_grad()
        outs = model(data[0].to(device=device))
        loss = loss_fn(data[1].to(device=device), outs) * weight[0]
        if len(weight) == 1:
            loss.backward()
        else:
            for step in range(1, steps):

                if (step == 1) or (hist == 0):
                    step_in = torch.concat(
                        (outs, data[int(step * 2)][:, N_in:].to(device=device)), 1
                    )
                    outs_old = outs
                elif (step > 1) and (hist == 1):
                    step_in = torch.concat(
                        (
                            outs,
                            data[int(step * 2)][:, N_in : (N_in + N_extra)].to(
                                device=device
                            ),
                            outs_old,
                        ),
                        1,
                    )
                    outs_old = outs
                else:
                    step_in = torch.concat(
                        (
                            outs,
                            data[int(step * 2)][:, N_in : (N_in + N_extra)].to(
                                device=device
                            ),
                            outs_old,
                            step_in[:, (N_in + N_extra) : -N_in],
                        ),
                        1,
                    )
                    outs_old = outs

                outs = model(step_in)
                outs = outs

                loss += (
                    loss_fn(data[int(step * 2 + 1)].to(device=device), outs)
                    * weight[step]
                )
            loss.backward()

        optimizer.step()
        torch.cuda.empty_cache()


def test_parallel_Dynamic(model, test_loader, device):
    mse = torch.nn.MSELoss()
    for data, label in test_loader:
        with torch.no_grad():
            outs = model(data.to(device=device))
            loss_val = mse(outs, label.to(device=device))
    return loss_val

# Transformers

In [10]:
import torch
from torch import nn
from torchvision.transforms import Resize
from einops.layers.torch import Rearrange
import copy


class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=9, patch_size=16, emb_size=768, img_size=(128, 144)):
        super().__init__()
        self.patch_size = patch_size
        self.pos_embed = nn.Parameter(
            torch.zeros(1, (img_size[0] * img_size[1]) // (patch_size**2) + 1, emb_size)
        )
        self.patch_to_embedding = nn.Linear(
            patch_size * patch_size * in_channels, emb_size
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_size))
        self.rearrange = Rearrange(
            "b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1=patch_size, p2=patch_size
        )

    def forward(self, x):
        b, _, _, _ = x.shape
        x = self.rearrange(x)
        x = self.patch_to_embedding(x)
        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embed
        return x


class VisionTransformer(nn.Module):
    def __init__(
        self,
        in_channels=9,
        patch_size=16,
        emb_size=768,
        num_layers=6,
        num_heads=8,
        output_channels=3,
        img_size=[128, 144],
    ):
        super().__init__()

        self.img_size = copy.copy(img_size)

        if img_size[0] % patch_size != 0:
            img_size[0] = (int(img_size[0] / patch_size) + 1) * patch_size

        if img_size[1] % patch_size != 0:
            img_size[1] = (int(img_size[1] / patch_size) + 1) * patch_size

        self.resize = Resize(img_size)
        self.patch_embedding = PatchEmbedding(
            in_channels, patch_size, emb_size, img_size
        )
        encoder_layer = nn.TransformerEncoderLayer(d_model=emb_size, nhead=num_heads)
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.to_pixel_values = nn.Sequential(
            nn.Linear(emb_size, patch_size * patch_size * output_channels),
            Rearrange(
                "b (h w) (p1 p2 c) -> b c (h p1) (w p2)",
                h=img_size[0] // patch_size,
                w=img_size[1] // patch_size,
                p1=patch_size,
                p2=patch_size,
                c=output_channels,
            ),
            Resize(img_size),
        )

    def forward(self, x):
        x = self.resize(x)
        x = self.patch_embedding(x)
        x = self.transformer_encoder(x)
        x = self.to_pixel_values(x[:, 1:, :])  # Skip the CLS token
        x = x[:, :, : self.img_size[0], : self.img_size[1]]
        return x


# Example usage
model = VisionTransformer(
    in_channels=9,
    patch_size=16,
    emb_size=768,
    num_layers=6,
    num_heads=8,
    output_channels=3,
    img_size=[*train_loader.dataset[0][0].shape[1:]],
)
input_data = torch.randn(6, 9, *train_loader.dataset[0][0].shape[1:])
output_data = model(input_data)
print(output_data.size())

torch.Size([6, 3, 119, 189])


In [11]:
model = VisionTransformer(
    in_channels=args["num_in"],
    patch_size=16,
    emb_size=768,
    num_layers=6,
    num_heads=8,
    output_channels=args["N_in"],
    img_size=[*train_loader.dataset[0][0].shape[1:]],
)

model = model.to(device)
# model.load_state_dict(torch.load("/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/saved_nets/Transformer_20_"+region+"_Test_in_"+str_in+"ext_"+str_ext+"_out"+str_in+"N_train_"+str(args["N_samples"]) +"_Lateral_Data_025_no_smooth.pt",map_location=torch.device(device)))
lam = args["lam"]
mse = torch.nn.MSELoss()
loss = lambda out, pred: mse(out, pred) * (1 - lam) + loss_KE_pointwise(out, pred) * lam

In [12]:
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
params = sum([np.prod(p.size()) for p in model_parameters])
print("Number of parameters: ", params)

for data in train_loader:
    print("Input shape: ", data[0].shape)
    out = model(data[0])
    print("Output shape: ", out.shape)
    break

Number of parameters:  35520000
Input shape:  torch.Size([6, 9, 119, 189])
Output shape:  torch.Size([6, 3, 119, 189])


In [13]:
# Not resetting optimizer
lr = 1e-4  # args["step_lrs"][0]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [14]:
step_weights = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
best_loss = torch.tensor(1e8)
args["epochs"] = 100
for epoch in range(args["epochs"]):
    train_sampler.set_epoch(int(epoch))
    train_parallel_Dynamic(
        model,
        train_loader,
        args["N_in"],
        args["N_extra"],
        args["hist"],
        loss,
        optimizer,
        args["steps"],
        step_weights,
        device,
    )

    v_loss = test_parallel_Dynamic(model, test_loader, device)

    print(
        "Epoch = {:2d}, Validation Loss = {:5.3f}".format(epoch + 1, v_loss), flush=True
    )

    if v_loss < best_loss:
        print("Achieved Best Validation Loss = {:5.3f}".format(v_loss))
        best_loss = v_loss
        print("Saving best model at epoch {0}".format(epoch + 1))
        torch.save(
            model.state_dict(),
            "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/saved_nets/"
            + "Transformer_best_steps_"
            + str(args["steps"])
            + "_"
            + args["region"]
            + "_Test_in_"
            + args["str_in"]
            + "ext_"
            + args["str_ext"]
            + "_out"
            + args["str_out"]
            + "N_train_"
            + str(args["N_samples"])
            + args["str_video"]
            + ".pt",
        )

    if (epoch + 1) % 10 == 0:
        print("Saving model at epoch {0}".format(epoch + 1))
        torch.save(
            model.state_dict(),
            "/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/saved_nets/"
            + "Transformer_epoch_"
            + str(epoch + 1)
            + "_steps_"
            + str(args["steps"])
            + "_"
            + args["region"]
            + "_Test_in_"
            + args["str_in"]
            + "ext_"
            + args["str_ext"]
            + "_out"
            + args["str_out"]
            + "N_train_"
            + str(args["N_samples"])
            + args["str_video"]
            + ".pt",
        )

Epoch =  1, Validation Loss = 0.722
Achieved Best Validation Loss = 0.722
Saving best model at epoch 1
Epoch =  2, Validation Loss = 0.631
Achieved Best Validation Loss = 0.631
Saving best model at epoch 2
Epoch =  3, Validation Loss = 0.566
Achieved Best Validation Loss = 0.566
Saving best model at epoch 3
Epoch =  4, Validation Loss = 0.520
Achieved Best Validation Loss = 0.520
Saving best model at epoch 4
Epoch =  5, Validation Loss = 0.479
Achieved Best Validation Loss = 0.479
Saving best model at epoch 5
Epoch =  6, Validation Loss = 0.449
Achieved Best Validation Loss = 0.449
Saving best model at epoch 6
Epoch =  7, Validation Loss = 0.423
Achieved Best Validation Loss = 0.423
Saving best model at epoch 7
Epoch =  8, Validation Loss = 0.401
Achieved Best Validation Loss = 0.401
Saving best model at epoch 8
Epoch =  9, Validation Loss = 0.382
Achieved Best Validation Loss = 0.382
Saving best model at epoch 9
Epoch = 10, Validation Loss = 0.363
Achieved Best Validation Loss = 0.363

In [15]:
# if args["save_model"]:
#     torch.save(model.state_dict(),'/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/saved_nets/' + 'Transformer_epoch_53_'+args["region"]+'_Test_in_' + args["str_in"] + 'ext_' + args["str_ext"] +'_out'+args['str_out']+'N_train_' + str(args["N_samples"]) + args["str_video"] + '.pt')